[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/25_time_series/04_feature_based_forecasting_and_backtesting.ipynb)

# Bài 04 — Dự báo bằng feature và backtesting

**🎯 Mục tiêu**
- Biến dự báo thành **học có giám sát**: feature lịch (thứ, Fourier mùa vụ năm, Tết), xu hướng, biến biết trước (khuyến mãi, lễ), trễ.
- **Rò rỉ** trong chuỗi thời gian và câu hỏi then chốt: *feature này có sẵn tại thời điểm dự báo không?*
- Vì sao **cross-validation ngẫu nhiên** đánh lừa; vì sao **cây không ngoại suy được xu hướng** và cách lai (hybrid).
- **Backtesting** nhiều điểm gốc (rolling origin); khoảng dự báo từ phần dư backtest.
- Mạng nơ-ron cho dự báo: khi nào giúp, khi nào không.
- Template model: dự báo chuỗi thời gian.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/25_time_series

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict

from ml_course.visualization import set_style

set_style()

In [ ]:
df = pd.read_csv("../../data/sample/sales_daily.csv", parse_dates=["date"]).set_index("date").asfreq("D")
s = df["sales"]
y_log = np.log(s.values)
n, H = len(s), 91
T = n - H                                                               # cùng holdout với bài 02–03 (Q4/2024)
mase_scale = np.mean(np.abs(s.values[:T][7:] - s.values[:T][:-7]))


def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))

## 1. Feature cho dự báo

Mỗi dòng = một ngày; target = log doanh số ngày đó. Feature phải **biết được tại thời điểm dự báo** cho mọi ngày trong tầm dự báo:

| Feature | Ý nghĩa | Có sẵn khi dự báo 91 ngày tới? |
|---|---|---|
| `t` (năm kể từ đầu) | xu hướng | ✅ |
| `dow_0..6` | mùa vụ tuần | ✅ |
| `sin_k, cos_k` ($k = 1..3$) | mùa vụ năm trơn (Fourier) | ✅ |
| `tet` | độ gần Tết | ✅ |
| `promo`, `holiday` | lịch khuyến mãi, ngày lễ | ✅ (đã lên kế hoạch) |
| `lag_1` | doanh số hôm qua | ❌ chỉ cho tầm 1 ngày |
| `lag_7` | cùng thứ tuần trước | ❌ chỉ cho tầm ≤ 7 ngày |

Fourier: $\sin(2\pi k\,\text{doy}/365.25)$, $\cos(\cdot)$ — vài cặp sin/cos đủ mô tả mùa vụ năm trơn với rất ít tham số (thay vì 365 biến giả).

In [ ]:
def calendar_features(index):
    F = pd.DataFrame(index=index)
    F["t"] = (index - pd.Timestamp("2022-01-01")).days / 365.25
    for d in range(7):
        F[f"dow_{d}"] = (index.dayofweek == d).astype(float)
    doy = index.dayofyear.to_numpy()
    for k in [1, 2, 3]:
        F[f"sin_{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
        F[f"cos_{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    F["tet"] = np.exp(-0.5 * ((doy - 30) / 7) ** 2)                      # đơn giản hoá: Tết quanh ngày thứ 30 của năm
    F["promo"] = df.loc[index, "promo"].to_numpy(float)
    F["holiday"] = df.loc[index, "is_holiday"].to_numpy(float)
    return F


F = calendar_features(s.index)
lin = LinearRegression().fit(F.iloc[:T], y_log[:T])
pred_lin = np.exp(lin.predict(F.iloc[T:]))
print(f"hồi quy tuyến tính trên log với feature lịch: MAE = {mae(s.values[T:], pred_lin):.2f} | MASE = {mae(s.values[T:], pred_lin) / mase_scale:.3f}")
print("(bài 02: Holt–Winters MASE 0.838 | bài 03: AR(14) + khuyến mãi/lễ MASE 0.738)")
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index[T - 60:], s.values[T - 60:], color="k", lw=0.8, label="thực tế")
ax.plot(s.index[T:], pred_lin, color="r", label="hồi quy tuyến tính + feature lịch")
ax.axvline(s.index[T], color="gray", ls=":")
ax.legend()
plt.show()

Mô hình tuyến tính đơn giản với **đúng feature** vượt xa mọi phương pháp trước: nó biết mùa vụ năm (Q4 thấp), lịch khuyến mãi, ngày lễ. Hệ số trên log có nghĩa trực tiếp: $e^{\beta_{\text{promo}}} - 1$ = % tăng doanh số khi khuyến mãi.

In [ ]:
coefs = pd.Series(lin.coef_, index=F.columns)
print(f"khuyến mãi: +{np.exp(coefs['promo']) - 1:.0%} | ngày lễ: +{np.exp(coefs['holiday']) - 1:.0%} | xu hướng: +{np.exp(coefs['t']) - 1:.1%}/năm")

## 2. Rò rỉ: "quá tốt để là thật"

Một feature phổ biến: trung bình 7 ngày gần nhất `rolling(7).mean()`. Nếu quên `shift`, nó **bao gồm chính ngày cần dự báo**. Kết hợp thêm trễ 1–6 ngày:

In [ ]:
for shift, label in [(0, "rolling(7).mean() — GỒM hôm nay (rò rỉ)"), (1, "shift(1).rolling(7).mean() — chỉ quá khứ")]:
    G = pd.DataFrame({"roll7": s.shift(shift).rolling(7).mean(), **{f"lag_{k}": s.shift(k) for k in range(1, 7)}}).iloc[8:]
    target = s.values[8:]
    model = LinearRegression().fit(G.iloc[:T - 8], target[:T - 8])
    print(f"{label:45s} MAE (dự báo 1 ngày) = {mae(target[T - 8:], model.predict(G.iloc[T - 8:])):6.2f} | "
          f"hệ số roll7 = {model.coef_[0]:.2f}, trễ 1..6 = {model.coef_[1:].round(2)}")

MAE = 0 với hệ số đúng bằng 7 và −1: mô hình đã "phát hiện" công thức $y_t = 7\cdot\text{roll7}_t - (y_{t-1} + \dots + y_{t-6})$ — nó **đọc đáp án** từ feature. Khi triển khai, `roll7` của ngày mai chưa tồn tại → mô hình vô dụng.

**Quy tắc:** với mỗi feature, hỏi *"tại thời điểm dự báo, với tầm $h$, giá trị này đã biết chưa?"*. Thống kê trượt luôn `shift(h)`; trễ phải ≥ $h$ (chiến lược **direct**) hoặc dự báo **đệ quy** (dùng chính dự báo làm trễ — sai số tích luỹ).

## 3. Cross-validation ngẫu nhiên đánh lừa

KFold xáo trộn đặt ngày test **xen giữa** các ngày train → mô hình chỉ cần **nội suy**. Dự báo thật là **ngoại suy** sang tương lai:

In [ ]:
rows = []
for name, make in [("Linear Regression", LinearRegression), ("HistGradientBoosting", lambda: HistGradientBoostingRegressor(random_state=0))]:
    oof = np.exp(cross_val_predict(make(), F.iloc[:T], y_log[:T], cv=KFold(5, shuffle=True, random_state=0)))
    future = np.exp(make().fit(F.iloc[:T], y_log[:T]).predict(F.iloc[T:]))
    rows.append({"mô hình": name, "MAE theo KFold ngẫu nhiên (trên train)": mae(s.values[:T], oof), "MAE thật trên tương lai (Q4/2024)": mae(s.values[T:], future)})
pd.DataFrame(rows).set_index("mô hình").round(2)

KFold ngẫu nhiên khen HistGB (MAE ~16) nhưng trên tương lai thật nó kém hẳn (MAE ~27). Nguyên nhân: **cây không ngoại suy được** — với `t` vượt quá giá trị lớn nhất trong train, cây trả về giá trị của lá cuối cùng → xu hướng tăng bị "đóng băng". KFold ngẫu nhiên không phát hiện được vì mọi giá trị `t` trong fold test đều nằm trong khoảng của train.

In [ ]:
t_grid = pd.date_range("2022-01-01", "2026-12-31", freq="D")
probe = pd.DataFrame(0.0, index=range(len(t_grid)), columns=F.columns)
probe["t"] = (t_grid - pd.Timestamp("2022-01-01")).days / 365.25
probe["dow_2"] = 1.0                                                    # cố định: thứ Tư, không mùa vụ năm, không khuyến mãi
hgb_full = HistGradientBoostingRegressor(random_state=0).fit(F.iloc[:T], y_log[:T])
plt.figure(figsize=(12, 4))
plt.plot(t_grid, np.exp(lin.predict(probe)), label="Linear: ngoại suy xu hướng")
plt.plot(t_grid, np.exp(hgb_full.predict(probe)), label="HistGB: phẳng sau dữ liệu train")
plt.axvspan(s.index[T], t_grid[-1], color="gray", alpha=0.15, label="ngoài vùng train")
plt.legend()
plt.title("Dự đoán theo t (giữ các feature khác cố định)")
plt.show()

**Cách lai (hybrid):** mô hình tuyến tính bắt **xu hướng**, cây học **phần dư** (mùa vụ, tương tác phi tuyến, khuyến mãi) — cây chỉ cần nội suy những thứ lặp lại.

## 4. Backtesting nhiều điểm gốc

Một holdout duy nhất rất nhiễu (bài 02). **Rolling-origin backtest**: lặp lại nhiều lần "train đến ngày $T_k$ → dự báo $h$ ngày tiếp theo", với $T_k$ tiến dần (cửa sổ mở rộng).

```
fold 1: [train ..........][test 28]
fold 2: [train ..................][test 28]
...
fold 8: [train ...................................][test 28]
```

In [ ]:
def backtest(forecast_fn, n_folds=8, horizon=28):
    errors = []
    for k in range(n_folds):
        end = n - (n_folds - k) * horizon
        errors.append(mae(s.values[end:end + horizon], forecast_fn(end, horizon)))
    return np.array(errors)


def seasonal_naive(end, h):
    return np.tile(s.values[end - 7:end], h // 7 + 1)[:h]


def linear_calendar(end, h):
    return np.exp(LinearRegression().fit(F.iloc[:end], y_log[:end]).predict(F.iloc[end:end + h]))


def hgb_calendar(end, h):
    return np.exp(HistGradientBoostingRegressor(random_state=0).fit(F.iloc[:end], y_log[:end]).predict(F.iloc[end:end + h]))


def hybrid(end, h):
    trend = LinearRegression().fit(F[["t"]].iloc[:end], y_log[:end])
    resid = y_log[:end] - trend.predict(F[["t"]].iloc[:end])
    rest = F.drop(columns="t")
    trees = HistGradientBoostingRegressor(random_state=0).fit(rest.iloc[:end], resid)
    return np.exp(trend.predict(F[["t"]].iloc[end:end + h]) + trees.predict(rest.iloc[end:end + h]))


bt = {name: backtest(fn) for name, fn in [("seasonal naive", seasonal_naive), ("HistGB + lịch", hgb_calendar),
                                          ("hybrid: tuyến tính (xu hướng) + HistGB (phần dư)", hybrid),
                                          ("Linear + lịch", linear_calendar)]}
scale_all = np.mean(np.abs(s.values[7:] - s.values[:-7]))
summary = pd.DataFrame({name: {"MAE trung bình": e.mean(), "độ lệch chuẩn giữa các fold": e.std(), "MASE": e.mean() / scale_all,
                               "fold tệ nhất": e.max()} for name, e in bt.items()}).T.round(3)
summary

In [ ]:
plt.figure(figsize=(11, 4))
for name, e in bt.items():
    plt.plot(range(1, 9), e, marker="o", label=name)
plt.xlabel("fold (điểm gốc tiến dần theo thời gian)")
plt.ylabel("MAE 28 ngày")
plt.legend(fontsize=8)
plt.show()
E = np.array(list(bt.values()))
winners = [list(bt)[j] for j in E.argmin(0)]
print("mô hình tốt nhất từng fold:", pd.Series(winners).value_counts().to_dict())

- Xếp hạng **khá ổn định** qua 8 điểm gốc: Linear + lịch tốt nhất ở hầu hết các fold và có fold tệ nhất nhẹ nhất; hybrid sửa được phần lớn điểm yếu ngoại suy của HistGB; seasonal naive kém nhất. Một fold riêng lẻ có thể đảo thứ tự — lý do cần nhiều điểm gốc.
- Dữ liệu này được sinh từ các thành phần nhân tính (xu hướng × tuần × năm × khuyến mãi) — đúng dạng log-tuyến tính → Linear thắng. Với dữ liệu thật có tương tác phức tạp (khuyến mãi khác nhau theo mùa, theo cửa hàng...), cây/boosting và mô hình **toàn cục** (một mô hình cho hàng nghìn chuỗi, như các giải M5) thường mạnh hơn.

### Mạng nơ-ron (neural forecasting) trong cùng khung backtest

Mạng nơ-ron có phải lựa chọn tốt hơn? Thử một MLP 2 lớp ẩn trên **cùng** feature lịch, cùng 8 điểm gốc, với 3 seed khởi tạo khác nhau. Thêm một biến thể "lai": hồi quy tuyến tính bắt xu hướng, MLP chỉ học phần dư (như mô hình hybrid ở trên):

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def mlp_calendar(end, h, seed=0, detrend=False):
    """MLP (32, 16) trên feature lịch; detrend=True: tuyến tính bắt xu hướng, MLP học phần dư."""
    X_all = F.drop(columns="t") if detrend else F
    target = y_log[:end]
    if detrend:
        trend = LinearRegression().fit(F[["t"]].iloc[:end], target)
        target = target - trend.predict(F[["t"]].iloc[:end])
    mlp = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-3, max_iter=3000, random_state=seed))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        pred = mlp.fit(X_all.iloc[:end], target).predict(X_all.iloc[end:end + h])
    if detrend:
        pred = pred + trend.predict(F[["t"]].iloc[end:end + h])
    return np.exp(pred)


neural = {}
for detrend, name in [(False, "MLP + lịch"), (True, "MLP học phần dư (xu hướng tuyến tính)")]:
    neural[name] = np.array([backtest(lambda end, h, sd=sd: mlp_calendar(end, h, sd, detrend)) for sd in range(3)])
rows_nn = [{"mô hình": "Linear + lịch", "MAE backtest": bt["Linear + lịch"].mean(), "MAE theo từng seed": "—"}]
for name, res in neural.items():
    rows_nn.append({"mô hình": name, "MAE backtest": res.mean(), "MAE theo từng seed": ", ".join(f"{m:.1f}" for m in res.mean(1))})
print(pd.DataFrame(rows_nn).set_index("mô hình").round(2).to_string())

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index[T - 60:], s.values[T - 60:], color="k", lw=0.8, label="thực tế")
ax.plot(s.index[T:], pred_lin, color="r", label="Linear + lịch")
for sd, style in [(0, "-"), (1, "--")]:
    ax.plot(s.index[T:], mlp_calendar(T, H, sd), color="C0", ls=style, lw=1, label=f"MLP + lịch (seed {sd})")
ax.plot(s.index[T:], mlp_calendar(T, H, 0, detrend=True), color="C2", lw=1, label="MLP học phần dư (seed 0)")
ax.axvline(s.index[T], color="gray", ls=":")
ax.legend(fontsize=8, ncol=2)
ax.set_title("Holdout Q4/2024: MLP trên feature thô lệch hẳn và đổi theo seed")
plt.show()

- MLP trên feature thô **thua xa** hồi quy tuyến tính và **không ổn định theo seed**: biến xu hướng `t` khi dự báo nằm **ngoài** vùng train, và mạng ReLU ngoại suy tuỳ tiện — cùng vấn đề với cây ở mục 3, chỉ là theo cách khác.
- Khử xu hướng trước (MLP chỉ học phần dư) sửa được phần lớn, nhưng vẫn không vượt mô hình tuyến tính trên **một** chuỗi ~1 000 ngày có cấu trúc log-tuyến tính.
- Neural forecasting thật sự đáng dùng khi có **nhiều chuỗi liên quan** (một mô hình *toàn cục* học chung cho hàng nghìn sản phẩm/cửa hàng — DeepAR, N-BEATS, Temporal Fusion Transformer), lịch sử dài, tần suất cao (theo phút, giờ) và nhiều biến ngoại sinh phức tạp; kiến trúc tuần tự: RNN/LSTM ([chương 30](../30_sequence_models/README.md)), 1D-CNN, Transformer ([chương 31](../31_transformers/README.md)). Dù dùng gì, luôn so với baseline và mô hình feature + boosting **trong cùng một backtest**.

## 5. Khoảng dự báo từ phần dư backtest

Ý tưởng (gần với **conformal prediction**, chương 37): thu thập sai số của các dự báo backtest **trước** holdout, lấy phân vị 2.5% và 97.5% làm biên.

In [ ]:
resid_bt = []
for k in range(12):                                                     # 12 fold × 28 ngày, đều TRƯỚC holdout
    end = T - (12 - k) * 28
    model_k = LinearRegression().fit(F.iloc[:end], y_log[:end])
    resid_bt.extend(y_log[end:end + 28] - model_k.predict(F.iloc[end:end + 28]))
q_lo, q_hi = np.quantile(resid_bt, [0.025, 0.975])
log_pred = lin.predict(F.iloc[T:])
lo, hi = np.exp(log_pred + q_lo), np.exp(log_pred + q_hi)
coverage = np.mean((s.values[T:] >= lo) & (s.values[T:] <= hi))
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index[T:], s.values[T:], color="k", lw=0.8, label="thực tế")
ax.plot(s.index[T:], np.exp(log_pred), color="r", label="dự báo")
ax.fill_between(s.index[T:], lo, hi, color="r", alpha=0.2, label="khoảng 95% (phân vị phần dư backtest)")
ax.legend()
plt.show()
print(f"phân vị phần dư log: [{q_lo:.3f}, {q_hi:.3f}] | độ phủ trên holdout = {coverage:.1%} | độ rộng TB = {np.mean(hi - lo):.0f}")

Độ phủ gần 95% với độ rộng hợp lý (so với khoảng của seasonal naive ở bài 02 rộng tới hàng trăm). Lưu ý: phân vị lấy từ phần dư **ngoài mẫu** (backtest), không phải phần dư trên train — phần dư trên train thường nhỏ hơn thực tế (lạc quan).

## 6. Template model: dự báo chuỗi thời gian

| Bước | Câu hỏi | Công cụ |
|---|---|---|
| Bài toán | tầm dự báo $h$? tần suất? quyết định dùng dự báo là gì? | — |
| Khám phá | xu hướng, mùa vụ (mấy chu kỳ?), sự kiện, thay đổi chế độ | plot, decomposition, ACF |
| Baseline | seasonal naive, drift | MASE |
| Feature | lịch, Fourier, sự kiện biết trước, trễ ≥ $h$, thống kê trượt `shift(h)` | pandas |
| Mô hình | thống kê (ETS, ARIMA) / hồi quy feature / boosting / lai / toàn cục / mạng nơ-ron (khi có nhiều chuỗi) | statsmodels, sklearn, LightGBM, PyTorch |
| Đánh giá | rolling-origin backtest, không xáo trộn | TimeSeriesSplit (+ `gap`) |
| Bất định | khoảng từ phần dư ngoài mẫu, quantile regression; kiểm tra độ phủ | — |
| Vận hành | train lại định kỳ, giám sát sai số & drift, cập nhật lịch sự kiện | chương 40 |

**Lỗi thường gặp:** rò rỉ (rolling không shift, scaler fit trên toàn chuỗi, target encoding dùng tương lai), CV ngẫu nhiên, thiếu baseline, chỉ một holdout, cây ngoại suy xu hướng, feature không có sẵn khi dự báo.

## 🧠 Kiểm tra nhanh

1. Vì sao `lag_1` không dùng được khi dự báo 28 ngày (chiến lược direct)? Có những lựa chọn nào?
2. Vì sao mô hình rò rỉ ở mục 2 đạt MAE = 0?
3. Vì sao KFold ngẫu nhiên đánh giá quá cao HistGB trên dữ liệu có xu hướng?
4. Hybrid tuyến tính + cây giải quyết vấn đề gì?
5. Vì sao lấy phân vị phần dư từ backtest thay vì phần dư trên train?
6. Khi nào mạng nơ-ron (neural forecasting) đáng dùng hơn mô hình feature + tuyến tính/boosting?

## 📌 Tóm tắt

- Dự báo = học có giám sát với feature **có sẵn tại thời điểm dự báo**; feature lịch + sự kiện biết trước thường mạnh nhất.
- Không CV ngẫu nhiên; backtest nhiều điểm gốc; cây không ngoại suy → khử xu hướng/lai.
- Khoảng dự báo từ phần dư ngoài mẫu; kiểm tra độ phủ.

## 📚 Tài liệu tham khảo

- Hyndman & Athanasopoulos — *Forecasting: Principles and Practice* (3rd ed.), ch. 5.10 (time series CV), ch. 7 & 12
- Makridakis, Spiliotis & Assimakopoulos (2022) — *M5 accuracy competition: Results, findings, and conclusions*
- Bergmeir, Hyndman & Koo (2018) — *A note on the validity of cross-validation for evaluating autoregressive time series prediction*